# Gross-to-Net Sales & Promotion Variance — Reproducible Analysis

A commercial-finance case study using **synthetic retail transactions** and **illustrative modeled assumptions**. The analysis reproduces the existing financial-export totals without claiming real promotional ROI. Run from the repository root.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
assert (ROOT/"powerbi_monthly_financials.csv").exists(), "Start Jupyter from repository root"
monthly=pd.read_csv(ROOT/"powerbi_monthly_financials.csv")
scenarios=pd.read_csv(ROOT/"powerbi_scenario_financials.csv")
bridge=pd.read_csv(ROOT/"variance_bridge.csv")
transactions=pd.read_csv(ROOT/"retail_transactions_SYNTHETIC.csv")
print(f"Transactions: {len(transactions):,}; month-category rows: {len(monthly):,}")

## 1. Gross-to-net and contribution-profit reconciliation

Net sales = gross sales − promotions − returns − trade spend. Contribution profit = net sales − COGS − variable selling expense. All amounts GBP.

In [ ]:
totals=monthly.select_dtypes("number").sum()
report=pd.Series({"Gross sales":totals.GrossSales_GBP,"Promotional discounts":totals.PromoDiscounts_GBP,"Returns":totals.Returns_GBP,"Trade spend":totals.TradeSpend_GBP,"Net sales":totals.NetSales_GBP,"COGS":totals.COGS_GBP,"Variable selling":totals.VariableSelling_GBP,"Contribution profit":totals.Contribution_GBP,"Plan contribution":totals.PlanContribution_GBP,"Variance vs plan":totals.Contribution_GBP-totals.PlanContribution_GBP})
assert abs(totals.GrossSales_GBP-totals.PromoDiscounts_GBP-totals.Returns_GBP-totals.TradeSpend_GBP-totals.NetSales_GBP)<0.15
assert abs(totals.NetSales_GBP-totals.COGS_GBP-totals.VariableSelling_GBP-totals.Contribution_GBP)<0.15
print(report.to_string(float_format=lambda v:f"£{v:,.2f}"))
print("Contribution margin: {:.2%}".format(totals.Contribution_GBP/totals.NetSales_GBP))

## 2. Contribution variance bridge

These are **modeled accounting differences versus plan**, not estimated causal effects.

In [ ]:
display(bridge)
assert abs(bridge.Amount_GBP.iloc[1:-1].sum()-(bridge.Amount_GBP.iloc[-1]-bridge.Amount_GBP.iloc[0]))<0.15
fig,ax=plt.subplots(figsize=(9,4));drivers=bridge.iloc[1:-1];ax.barh(drivers.Driver,drivers.Amount_GBP/1000);ax.axvline(0,color="gray",linewidth=0.8);ax.set_xlabel("Effect on contribution vs plan (£000s)");ax.set_title("Modeled plan-to-actual variance drivers");plt.tight_layout();plt.show()

## 3. Category and monthly trends

In [ ]:
categories=monthly.groupby("Category",as_index=False)[["NetSales_GBP","Contribution_GBP","PlanContribution_GBP"]].sum()
categories["ContributionMargin"]=categories.Contribution_GBP/categories.NetSales_GBP
categories["Variance_GBP"]=categories.Contribution_GBP-categories.PlanContribution_GBP
display(categories.round(3))
months=monthly.groupby("Month",as_index=False)[["NetSales_GBP","Contribution_GBP"]].sum()
months["MarginPct"]=months.Contribution_GBP/months.NetSales_GBP*100
ax=months.plot(x="Month",y="MarginPct",marker="o",figsize=(10,3.5),legend=False,title="Modeled monthly contribution margin");ax.set_ylabel("Margin (%)");plt.xticks(rotation=40);plt.tight_layout();plt.show()

## 4. Promotion scenario decisions

The scenario table uses documented illustrative changes in discount rate and volume. These are **not empirical elasticity estimates**.

In [ ]:
display(scenarios[["Scenario","Contribution_GBP","ContributionMargin","ContributionChange_GBP","DiscountRate","VolumeUplift"]].round(3))
ax=scenarios.plot.bar(x="Scenario",y="Contribution_GBP",figsize=(10,4),legend=False,title="Modeled contribution profit by promotion scenario");ax.set_ylabel("Contribution profit (£)");plt.xticks(rotation=15);plt.tight_layout();plt.show()
best=scenarios.loc[scenarios.Contribution_GBP.idxmax()]
print(f"Highest modeled contribution: {best.Scenario}, £{best.Contribution_GBP:,.0f} ({best.ContributionChange_GBP:+,.0f} vs baseline)")

## Recommendation and limitations

Under the current assumptions, the **Balanced offer** produces the highest contribution profit, while Margin protection generates the highest contribution margin percentage. Test discount/volume sensitivity before using this for a real decision.

The transaction-level discount rates are synthetic, and the monthly finance plan, returns, trade spend, COGS, variable selling costs and scenarios include modeling assumptions; they are not audited actuals. See the original README for the methodology and limitations.